# Task 0 — Setup and workflow check

**ML for Bioprocesses · Week 1 lab · 30–40 minutes · pass/fail**

This task teaches you no machine learning. It walks you once, end to end, through the routine you will
repeat every week of the course:

1. open the notebook and save your own copy,
2. run the setup cell,
3. work — swapping driver and navigator,
4. Restart & Run All, download, upload.

Doing it now, on easy material, means that in Week 1 your attention goes to the fermentation data and
not to the menus.

**You pass** when your pair has uploaded this notebook to the LMS, run top to bottom from a fresh
session, with no errors. Your answers to the questions are not graded for correctness.

## Part A — Get your own copy (account holder drives)

The partner who holds the pair's Google account is the **driver** for Parts A and B.

**A1. Run-anyway warning.** When you first run a cell, Colab may warn that this notebook was not authored
by Google. That is true: it comes from the course repository. Choose *Run anyway*. The general rule is:
run code only from sources you trust, and read what it does. In this course, the source of everything you
are given is published.

**A2. Save a copy.** *File → Save a copy in Drive.* A copy opens in a new tab. **Close the original tab and
work only in the copy.** Edits made in the original are not saved anywhere and disappear when you close it.
This is the most common way beginners lose work in Colab.

**A3. Rename** the copy (click the file name at the top left): `Task0_<surname1>_<surname2>`.

**A4. Share** the copy with your partner (*Share* button, top right, add their e-mail address).
*Viewer* access is enough.

**A5. Partner check.** The partner opens the shared link on **their own** phone or laptop and confirms
they can see it. If the account holder is ever absent, this is how the partner reaches the pair's work.

Fill in, by editing this cell (double-click it):

- Partner 1 (account holder): …
- Partner 2: …
- Partner 2 could open the shared copy on their own device (yes/no): …

## Part B — Setup

The next cell is the **setup cell**. Every notebook in this course starts with one. It installs the course
package `bioml` and imports the libraries used below. Run it at the start of **every** session: Colab
forgets everything when a session ends or disconnects.

To run a cell: click inside it and press **Shift + Enter**.

In [ ]:
# ---- Setup cell: run first, every session --------------------------------------------------------
import subprocess  # lets Python run a command, here the package installer pip
import sys  # information about the running Python, including whether we are in Colab

# The exact bioml release this course uses, pinned to one tag so Week 8 behaves like Week 1.
# Change only BIOML_TAG; the other two values are derived from it so they cannot disagree.
BIOML_TAG = "v1.0.2"
BIOML_VERSION = BIOML_TAG.lstrip("v")
BIOML_SPEC = f"bioml @ git+https://github.com/JohanAPersson/mlbio-course@{BIOML_TAG}"

# Colab starts every session without bioml installed, so we install it there. On a computer where it is
# already installed (for example in a university computer room) this step is skipped.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", BIOML_SPEC], check=True)

import bioml  # the course package: loading data, splitting, plotting
import matplotlib.pyplot as plt  # plotting
import pandas as pd  # tables of data ("DataFrames")

print(f"Python  {sys.version.split()[0]}")
print(f"pandas  {pd.__version__}")
print(f"bioml   {bioml.__version__}")
print(f"Colab   {IN_COLAB}")

# Stop here, loudly, if the wrong version of bioml was installed.
assert bioml.__version__ == BIOML_VERSION, (
    f"Expected bioml {BIOML_VERSION}, got {bioml.__version__}. Tell the instructor."
)

# Download the course data into ./data/data-v2 (about 20 MB). Files already present are skipped,
# so re-running this cell is cheap.
bioml.ensure_data()

## Part B (continued) — Load the data and check it

The next cell loads the process-variable table: one row per time point, for all 100 batches. It is
about 20 MB and loads in seconds.

In [ ]:
# Load the process-variable table as a pandas DataFrame. bioml fetches the course data the first time.
pv = bioml.load_process_vars()

# Column names, exactly as they appear in the file. Copy them from here rather than retyping them.
TIME = "Time (h)"
BATCH = "Batch ID"
VOLUME = "Vessel Volume(V:L)"

# .shape gives (number of rows, number of columns).
print(f"rows x columns: {pv.shape}")
# .nunique() counts distinct values; distinct batch numbers = number of batches.
print(f"batches:        {pv[BATCH].nunique()}")

**Check before you trust.** The next cell does not compute anything new. It tests the data against
things we already know: there are 100 batches, and the volume of a 100,000 L vessel must lie somewhere
below 100,000 L. If any check fails, the cell stops with an error message.

This habit matters more than it looks. The original file for this dataset contains a column labelled
`Batch ID` that does not hold batch numbers. It was found by exactly this kind of check.

In [ ]:
# Each assert stops the notebook with the message if its condition is False.
assert pv[BATCH].nunique() == 100, "Expected 100 batches."
assert VOLUME in pv.columns, f"Column {VOLUME!r} not found."
assert pv[VOLUME].between(4.0e4, 1.0e5).all(), "Volume outside the plausible range for this vessel."
print("All data checks passed.")

---
## 🔄 SWAP NOW

**Partner 2 takes the keyboard** and drives Parts C, D and E. From Week 1 onward you swap every 30
minutes; this is the first swap.
---

## Part C — Predict before you run

Every week, before you run a cell whose result matters, you write down what you expect and why — in a
text cell directly above it. A prediction that cannot turn out wrong is not a prediction: "the plot
will show the volume" earns nothing; "the volume will …, because …" does.

The next cell plots the vessel volume against time for batch 1. This is a **fed-batch** process.

**Your prediction** (edit this cell — one sentence, with a reason):

…

In [ ]:
# Keep only the rows belonging to batch 1. The expression inside [...] is True for those rows.
b1 = pv[pv[BATCH] == 1]

# One line plot: time on the x-axis, volume on the y-axis.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(b1[TIME], b1[VOLUME])
ax.set_xlabel("time (h)")
ax.set_ylabel("vessel volume (L)")
ax.set_title("Batch 1 — vessel volume")
plt.show()

**Compare** (edit this cell): Was your prediction right? If anything in the curve surprised you, which
physical operation on a fed-batch fermenter could explain it? One or two sentences.

…

## Part D — The notebook remembers what you ran, not what you see

This part shows the single most important rule for working with notebooks.

**D1.** Run the next three cells in order, once each.

In [ ]:
# Start a counter at zero.
n_feed_additions = 0

In [ ]:
# Record one feed addition by adding 1 to the counter.
n_feed_additions = n_feed_additions + 1

In [ ]:
# Show the current value of the counter.
print(f"feed additions recorded: {n_feed_additions}")

**D2.** Now run **only the middle cell** (the one that adds 1) a second time, then run the print cell
again. **Before you do**, predict what it will print:

…

**D3.** What happened? The code on your screen has not changed, yet the answer has. The notebook keeps a
memory (the *kernel*) of everything you have run, in the order you ran it — not in the order the cells
appear on the page. With silent errors this is dangerous: re-running cells out of order can give you a
different wrong answer, or an accidental right one, with no way to tell which.

**D4.** Run the check cell below. If you did D2, it fails. That is intended.

**D5.** Now fix it the only reliable way: **Runtime → Restart session and run all.** Colab clears its
memory and runs every cell from the top. Wait for it to finish, and look at the check cell again.

**The course rule:** everything you submit, every prediction we assess, and everything you show at an
oral checkpoint must come from *Restart session and run all*.

In [ ]:
# This checks the counter against what a clean top-to-bottom run produces (exactly one addition).
assert n_feed_additions == 1, (
    f"The counter is {n_feed_additions}, not 1: cells were run out of order. "
    "Use Runtime -> Restart session and run all."
)
print("Counter is 1, as a clean top-to-bottom run gives.")

## Part E — The AI assistant

Colab has a built-in AI assistant (the Gemini button). You may use it, or any other tool, freely in this
course. What is assessed is the following, stated here once and repeated at every oral checkpoint:

> You must be able to explain what any function you called does, why you chose it over the alternative,
> and what would change if you had chosen differently. The source of everything you were given is
> published. "I don't know what it does" is not available to you.

Try it now if you like: ask the assistant what `pv[pv[BATCH] == 1]` does, then check whether its answer
matches the comment in that cell.

## Part F — Submit

1. **Runtime → Restart session and run all.** Wait until every cell has finished.
2. Scroll through. The numbers in brackets to the left of the code cells must run **1, 2, 3, …** in order,
   with no gaps and no errors. A gap or a repeated number means cells were run by hand after the
   restart — go back to step 1.
3. **File → Download → Download .ipynb.**
4. Upload the downloaded file to the LMS under *Task 0*.

Done. This is the routine for every week of the course.

In [ ]:
# If this line prints, every cell above ran without an error stopping the notebook.
print("Reached the end of the notebook without errors. Check the bracket numbers, then download.")